# Evaluate the Fine-Tuned LoRA Adapter (Task #6)

**Task #6** of the weekend plan (Sunday AM-1): score the **r=16** adapter trained on Saturday against the locked 50-question eval set, using the *same* ROUGE + Gemini-judge pipeline as the baseline.

This notebook loads the **base** Mistral-7B in 4-bit, layers the trained LoRA adapter on top, runs `scripts/05_eval_finetuned.py` over `data/eval/eval.jsonl`, and writes `results/runs/r16/`.

**Runtime budget**: ~30-45 min on a free Colab T4 (50 generations @ 512 tokens + judge calls). You kept `EVAL_MAX_NEW_TOKENS=512`.

**Before you run this notebook, make sure:**
1. Runtime → Change runtime type → **T4 GPU** is selected.
2. The r=16 adapter is on Drive at `/MyDrive/LLM_Finetune/models/adapters/r16/` (Saturday's train notebook put it there).
3. `HF_TOKEN` **and** `GEMINI_API_KEY` are set in Colab Secrets (🔑 left sidebar). `HF_TOKEN` downloads the gated base; `GEMINI_API_KEY` powers the LLM-judge.
4. The latest commits are pushed to GitHub — Colab pulls a fresh copy below.

## 1. Confirm GPU is available

Should print a T4 with ~15 GB VRAM. If `nvidia-smi` fails or shows no GPU, switch the runtime type and restart.

In [ ]:
!nvidia-smi

## 2. Clone the project

Replace `<your-username>/LLM_Finetune` with your actual repo. For a private repo, use `https://<token>@github.com/<you>/LLM_Finetune.git` with a fine-grained PAT that has read access.

In [ ]:
!git clone https://github.com/<your-username>/LLM_Finetune.git
%cd LLM_Finetune

## 3. Install dependencies

Same install as the training notebook — we need **`unsloth`** to load the 4-bit base the adapter was trained on, plus our `requirements.txt` for `rouge_score`, `google-genai`, `python-dotenv`, etc.

**Why we do NOT `pip install -U trl peft accelerate bitsandbytes`**: Unsloth's compiled cache is generated against a *specific* TRL version. Upgrading TRL afterwards breaks Unsloth's monkey-patches. Let Unsloth pin its own deps.

Install takes ~2 minutes.

In [ ]:
%%capture
# NO `-U` on trl/peft/accelerate/bitsandbytes — that would break Unsloth's
# compiled cache. unsloth's own setup.py is the source of truth for pins.
!pip install -q unsloth datasets
!pip install -q -r requirements.txt

## 4. Load secrets

We read tokens from the Colab Secrets panel into env vars.
- `HF_TOKEN` — required to download the gated base Mistral weights.
- `GEMINI_API_KEY` — required for the LLM-judge (`gemini-3.1-flash-lite`, free tier). `src/eval/llm_judge.py` reads it from the environment.

(No W&B here — evaluation isn't logged as a training run.)

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"]               = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GEMINI_API_KEY"]         = userdata.get("GEMINI_API_KEY")

print(f"HF_TOKEN:        {'set' if os.environ.get('HF_TOKEN') else 'MISSING'}")
print(f"GEMINI_API_KEY:  {'set' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")

## 5. Mount Google Drive and redirect output dirs to Drive

**Two reasons we mount Drive here:**
1. **Read the adapter.** The r=16 adapter (161 MB, gitignored) lives only on Drive — not in the GitHub repo. We symlink `models/adapters/` to the Drive folder so `models/adapters/r16` resolves to it.
2. **Persist results.** Symlinking `results/runs/` to Drive means `results/runs/r16/` survives a runtime restart.

Same symlink trick as the training notebook. You'll get a Drive auth popup the first time — click through and allow all permissions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, pathlib

DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/LLM_Finetune")
(DRIVE_ROOT / "models/adapters").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "results/runs").mkdir(parents=True, exist_ok=True)

# Replace local output dirs with symlinks pointing INTO Drive, so that:
#   - reads from models/adapters/r16 find the adapter on Drive, and
#   - writes to results/runs/r16 land on Drive (survive runtime restart).
# We always recreate the symlinks so re-running this cell is safe.
for local_path_str, drive_path in [
    ("/content/LLM_Finetune/models/adapters", DRIVE_ROOT / "models/adapters"),
    ("/content/LLM_Finetune/results/runs",    DRIVE_ROOT / "results/runs"),
]:
    local_path = pathlib.Path(local_path_str)
    if local_path.is_symlink():
        local_path.unlink()
    elif local_path.exists():
        shutil.rmtree(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(drive_path, local_path)
    print(f"  {local_path} -> {drive_path}")

print("\nDrive mounted and output dirs symlinked.")

## 6. Verify the adapter is reachable

Quick sanity check before spending GPU time: confirm `models/adapters/r16/` resolves (through the symlink) to the adapter on Drive and contains the LoRA weights.

In [ ]:
import os

ADAPTER = "models/adapters/r16"
print(f"adapter dir exists?  {os.path.isdir(ADAPTER)}")
if os.path.isdir(ADAPTER):
    !ls -lh {ADAPTER}
    assert os.path.exists(f"{ADAPTER}/adapter_model.safetensors"), \
        "adapter_model.safetensors missing — is the Drive path correct?"
    print("\nAdapter looks good — ready to eval.")
else:
    print("[STOP] Adapter not found. Re-run section 5 and confirm the r=16")
    print("       adapter is at /MyDrive/LLM_Finetune/models/adapters/r16/")

## 7. Run the evaluation

This runs `scripts/05_eval_finetuned.py`, which:
1. Loads the Unsloth 4-bit base + layers the r=16 LoRA adapter on top (two-step PEFT load).
2. Generates an answer for each of the 50 eval questions (greedy, 512 tokens — same settings as the baseline).
3. Scores each with ROUGE-1/L + exact-match (local) and the Gemini judge (1–5).
4. Writes `results/runs/r16/per_example.csv` + `results/runs/r16/results.json` — symlinked to Drive.

**Runtime**: ~30–45 min on a T4. Don't close the tab.

In [ ]:
!python scripts/05_eval_finetuned.py --adapter models/adapters/r16 --run-name r16

## 8. Inspect the results

Print the summary JSON (headline ROUGE + judge numbers, plus per-category / per-difficulty breakdowns) and confirm both output files landed on Drive.

In [ ]:
import json

with open("results/runs/r16/results.json") as f:
    summary = json.load(f)

agg = summary["aggregate"]
print("=== r16 fine-tuned — aggregate ===")
print(f"  rouge1:      {agg['rouge1']:.4f}   (baseline 0.132)")
print(f"  rougeL:      {agg['rougeL']:.4f}   (baseline 0.107)")
print(f"  exact_match: {agg['exact_match']:.4f}")
if agg.get("judge_score") is not None:
    print(f"  judge_score: {agg['judge_score']:.2f} / 5   (baseline 2.28)")

print("\nFiles on Drive:")
!ls -lh results/runs/r16/

## 9. Commit the results back to git

`results/runs/r16/results.json` and `per_example.csv` are small and reproducible — they belong in git so the README results table has an audit trail. (The adapter itself stays gitignored.)

**You git-push from your local machine** (to keep the history free of `colab` author commits): copy `results/runs/r16/results.json` + `per_example.csv` from your Drive folder into your local repo, then `git add` + `git commit` + `git push`. After that, fill the `+ LoRA r=16` row in README §6 with the numbers from section 8.